# Project 7: Stream Analysis — Algorithms for Massive Datasets

Downloading the dataset

In [33]:
import os,glob,zipfile, random

random.seed(42) #will be used later on

try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
except ImportError:
    pass  # not on Colab — kaggle will find ~/.kaggle/kaggle.json automatically

ZIP = "new-york-times-articles-comments-2020.zip"

if not os.path.exists("data"):
    !pip install -q kaggle
    if not os.path.exists(ZIP):
        !kaggle datasets download -q -d benjaminawd/new-york-times-articles-comments-2020
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("data")
    print("Dataset downloaded and extracted.")
else:
    print("Dataset already present, skipping.")

for path in sorted(glob.glob("data/**/*", recursive=True)):
    if os.path.isfile(path):
        print(f"{os.path.getsize(path)/1e6:10.1f} MB   {path}")

Dataset already present, skipping.
       7.9 MB   data\nyt-articles-2020.csv
    3066.9 MB   data\nyt-comments-2020.csv
     303.3 MB   data\nyt-comments-part0.csv
     303.5 MB   data\nyt-comments-part1.csv
     302.0 MB   data\nyt-comments-part2.csv
     309.7 MB   data\nyt-comments-part3.csv
     309.0 MB   data\nyt-comments-part4.csv
     309.9 MB   data\nyt-comments-part5.csv
     308.5 MB   data\nyt-comments-part6.csv
     311.6 MB   data\nyt-comments-part7.csv
     310.9 MB   data\nyt-comments-part8.csv
     298.6 MB   data\nyt-comments-part9.csv
       1.9 MB   data\test.csv
       6.0 MB   data\train.csv


In [34]:
import csv

PATH = "data/nyt-comments-2020.csv"

with open(PATH, newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    print(f"{len(header)} columns:\n")
    for col in header:
        print("  ", col)

23 columns:

   commentID
   status
   commentSequence
   userID
   userDisplayName
   userLocation
   userTitle
   commentBody
   createDate
   updateDate
   approveDate
   recommendations
   replyCount
   editorsSelection
   parentID
   parentUserDisplayName
   depth
   commentType
   trusted
   recommendedFlag
   permID
   isAnonymous
   articleID


## Input layer

checking items in each row

In [35]:
PATH = "data/nyt-articles-2020.csv"

with open(PATH, newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    print(f"{len(header)} columns:\n")
    for col in header:
        print("  ", col)

11 columns:

   newsdesk
   section
   subsection
   material
   headline
   abstract
   keywords
   word_count
   pub_date
   n_comments
   uniqueID


In [36]:
# SUBSAMPLE flag

SUBSAMPLE = True
SUBSAMPLE_SIZE = 500000

if SUBSAMPLE:
    limit = SUBSAMPLE_SIZE
else:
    limit = None

Making a reader for the file - the `DictReader` reads lazily - one row at a time on demand. So the whole file is never loaded in memory, the reader grabs one row, yield passes the one line, which means only ever one row is loaded in memory

In [37]:
def comment_stream(path, limit):
    with open(path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
                if limit is not None and i >= limit:
                    break
                else:
                    yield row

# Algorithms
## Bloom filter

In [38]:
class BloomFilter:
    
    def __init__(self, size, num_hashes):
        self.size = size
        self.num_hashes = num_hashes
        self.bits = [0] * size
    
    def _hashes(self, item):
        values = []
        for i in range(self.num_hashes):
            #this makes it so I can get n hash functions from the same one - the hashing doesnt actually change, the item does in a determined way
            combine = f"{i}_{item}"
            h = hash(combine)
            position = abs(h) % self.size
            values.append(position)
        return values

    def add(self, item):
        values = self._hashes(item)
        for i in values:
            self.bits[i] = 1

    def check(self, item):
        present = True
        values = self._hashes(item)
        for i in values:
            if self.bits[i] == 0:
                present = False
        return present

## Flajolet-Martin

In [39]:
from statistics import median

class FlajoletMartin:
    def __init__(self, num_hashes):
        self.num_hashes = num_hashes
        self.max_tails = [0]*num_hashes

    def _tail_length(self, x):
        if x == 0:
            return 0
        tail_length = (x & -x).bit_length() - 1 #bitwise AND isolates lowest bit in binary expression. the bit is included, so -1 to get number of trailing zeros
        return tail_length
        

    def add(self, item):
        for i in range(self.num_hashes):
            combine = f"{i}_{item}"
            h = hash(combine)
            tail_length = self._tail_length(h)
            if tail_length > self.max_tails[i]:
                self.max_tails[i] = tail_length
        

    def estimate(self, group_size):
        estimates = []
        for r in self.max_tails:
            estimates.append(2**r)
        group_medians = []
        for start in range(0, len(estimates), group_size):
            group = estimates[start : start + group_size]
            m = median(group)
            group_medians.append(m) 
        return sum(group_medians)/len(group_medians)

## AMS
### Section lookup table

In [40]:
def load_sections(path):  #path of articles file
    section_dict = {}

    with open(path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            key = row['uniqueID']
            value = row['section']
            section_dict[key] = value
        return section_dict

### AMS 

In [41]:
class AMS:
    def __init__(self, num_vars):
        self.num_vars = num_vars
        self.vars_element = [None]*num_vars 
        self.vars_count = [0]*num_vars
        self.t = 0 #number of stream elements seen so far

    def add(self, element):
        self.t += 1
        for j in range(self.num_vars):
            if random.random() < 1/self.t: #selects randomly wether to keep the current counted element or to switch to a new one
                self.vars_element[j] = element
                self.vars_count[j] = 1
            elif element == self.vars_element[j]: #still counting same element, if it is a match:
                self.vars_count[j] += 1
                
    def estimate(self):
        res = []
        for j in range(self.num_vars):
            est = self.t * (2 * self.vars_count[j] - 1)
            res.append(est)
        return sum(res)/len(res)

## Running the algorithms on data stream

In [63]:
bloom = BloomFilter(size = 1000000, num_hashes = 3)
fm = FlajoletMartin(num_hashes = 20)
ams = AMS(num_vars=50)
section_of = load_sections("data/nyt-articles-2020.csv")

for row in comment_stream("data/nyt-comments-2020.csv", limit=limit):
    user = row['userID']
    article = row['articleID']
    section = section_of.get(article, "UNKNOWN")

    bloom.add(article)
    fm.add(user)
    ams.add(section)

In [64]:
print("Bloom — checked article seen:", bloom.check(article))   # last article, should be True
print("FM — estimated distinct users:", fm.estimate(group_size=5))
print("AMS — estimated F2 of sections:", ams.estimate())

Bloom — checked article seen: True
FM — estimated distinct users: 65536.0
AMS — estimated F2 of sections: 64791800000.0


In [68]:
users_set = set()
section_counts = {}
for row in comment_stream("data/nyt-comments-2020.csv", limit=limit):
    users_set.add(row['userID'])
    section = section_of.get(row['articleID'], "UNKNOWN")
    section_counts[section] = section_counts.get(section, 0) + 1
distinct_users = len(users_set)
true_f2 = sum(c*c for c in section_counts.values()) 
print(distinct_users, true_f2)

91437 62286396546
